In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType
import re

spark = SparkSession.builder.appName("Employee_ETL").getOrCreate()

# Read CSV Function
def read_csv(path):
    return spark.read.format("csv").option("header", True).option("inferSchema", True).load(path)

# Write CSV Function
def write_csv(df, path, mode="overwrite"):
    df.coalesce(1).write.mode(mode).option("header", True).csv(path)

# Step 5: CamelCase → snake_case function
def camel_to_snake(text):
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', text)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

# Register UDF
camel_to_snake_udf = udf(camel_to_snake, StringType())

# Rename all columns to snake_case
def convert_columns_to_snake(df):
    for column in df.columns:
        new_column = camel_to_snake(column)
        df = df.withColumnRenamed(column, new_column)
    return df